In [1]:
import os
import glob
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from torchvision.ops import roi_align

# ── Config ────────────────────────────────────────────────────────────────────
FEATURE_DIRS = {
    "train": ["/kaggle/input/datasets/kavishgrover/vitfeats/vit_features/train"],
    "val"  : ["/kaggle/input/datasets/kavishgrover/vitfeats/vit_features/val"],
    "test" : ["/kaggle/input/datasets/kavishgrover/vitfeats/vit_features/test"],
}
ACTIONS_JSON = {
    "train": "/kaggle/input/datasets/kavishgrover/bdd-oia/train_25k_images_actions.json",
    "val"  : "/kaggle/input/datasets/kavishgrover/bdd-oia/val_25k_images_actions.json",
    "test" : "/kaggle/input/datasets/kavishgrover/bdd-oia/test_25k_images_actions.json",
}
REASONS_JSON = {
    "train": "/kaggle/input/datasets/kavishgrover/bdd-oia/train_25k_images_reasons.json",
    "val"  : "/kaggle/input/datasets/kavishgrover/bdd-oia/val_25k_images_reasons.json",
    "test" : "/kaggle/input/datasets/kavishgrover/bdd-oia/test_25k_images_reasons.json",
}

TOP_K        = 10
NUM_ACTIONS  = 4
NUM_REASONS  = 21
LAMBDA       = 1.0
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 50
LR_STEP      = 10
BATCH_SIZE   = 16
NUM_WORKERS  = 2
PATIENCE     = 7
SAVE_PATH    = "/kaggle/working/vit_model_best.pt"
LOG_PATH     = "/kaggle/working/vit_training_log.json"
PATCH_SIZE   = 14
GRID_SIZE    = 16
VIT_DIM      = 768
device       = "cuda" if torch.cuda.is_available() else "cpu"

# ── Label loading ─────────────────────────────────────────────────────────────
def load_label_lookups(split):
    with open(ACTIONS_JSON[split]) as f:
        actions_data = json.load(f)
    with open(REASONS_JSON[split]) as f:
        reasons_data = json.load(f)

    id_to_filename = {img["id"]: img["file_name"] for img in actions_data["images"]}

    action_lookup = {}
    for ann in actions_data["annotations"]:
        cat = ann["category"]
        if len(cat) == 5 and cat[4] == 1:
            continue
        action_lookup[id_to_filename[ann["id"]]] = torch.tensor(cat[:4], dtype=torch.float32)

    reason_lookup = {}
    for entry in reasons_data:
        reason_lookup[entry["file_name"]] = torch.tensor(entry["reason"], dtype=torch.float32)

    return action_lookup, reason_lookup

# ── Dataset ───────────────────────────────────────────────────────────────────
class BDDOIADataset(Dataset):
    def __init__(self, split):
        self.action_lookup, self.reason_lookup = load_label_lookups(split)

        self.pt_files = []
        for d in FEATURE_DIRS[split]:
            if os.path.exists(d):
                self.pt_files += glob.glob(os.path.join(d, "*.pt"))

        self.pt_files = [
            f for f in self.pt_files
            if self._get_fname(f) in self.action_lookup
            and self._get_fname(f) in self.reason_lookup
        ]
        print(f"[{split}] {len(self.pt_files)} samples loaded")

    def _get_fname(self, pt_path):
        return os.path.basename(pt_path).replace(".pt", ".jpg")

    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        data  = torch.load(self.pt_files[idx], weights_only=True)
        fname = self._get_fname(self.pt_files[idx])
        return (
            data["cls_token"].float(),                              # (768,)
            data["patch_tokens"].float(),                           # (256, 768)
            self.action_lookup[fname],                              # (4,)
            self.reason_lookup[fname],                              # (21,)
            torch.as_tensor(data["boxes"], dtype=torch.float32),   # (N, 4)
        )

def collate_fn(batch):
    return (
        torch.stack([item[0] for item in batch]),  # cls_tokens    (B, 768)
        torch.stack([item[1] for item in batch]),  # patch_tokens  (B, 256, 768)
        torch.stack([item[2] for item in batch]),  # action_labels (B, 4)
        torch.stack([item[3] for item in batch]),  # reason_labels (B, 21)
        [item[4] for item in batch],               # boxes list, variable N
    )

# ── RoI Align on patch grid ───────────────────────────────────────────────────
def extract_roi_features(patch_tokens, boxes):
    """
    patch_tokens : (256, 768)  — flat patch tokens from DINOv2
    boxes        : (N, 4)      — YOLO boxes in 224×224 pixel space, float32
    Returns      : (N, 768, 7, 7)
    """
    # Correct spatial reshape:
    # (256, 768) → (16, 16, 768) → (768, 16, 16) → (1, 768, 16, 16)
    feat_map = patch_tokens.view(GRID_SIZE, GRID_SIZE, VIT_DIM)\
                           .permute(2, 0, 1)\
                           .unsqueeze(0)\
                           .float()               # (1, 768, 16, 16)

    # Ensure boxes are float32 on correct device
    boxes = torch.as_tensor(boxes, device=patch_tokens.device, dtype=torch.float32)
    N     = boxes.shape[0]

    if N == 0:
        boxes = torch.tensor([[0.0, 0.0, 224.0, 224.0]],
                             device=patch_tokens.device, dtype=torch.float32)
        N = 1

    batch_idx = torch.zeros(N, 1, device=patch_tokens.device, dtype=torch.float32)
    roi_boxes = torch.cat([batch_idx, boxes], dim=1)  # (N, 5)

    local_feats = roi_align(
        feat_map,
        roi_boxes,
        output_size=(7, 7),
        spatial_scale=1.0 / PATCH_SIZE,   # 1/14 maps 224px coords to 16×16 grid
        aligned=True,
    )  # (N, 768, 7, 7)

    return local_feats

# ── Model ─────────────────────────────────────────────────────────────────────
class AISM(nn.Module):
    def __init__(self, k=10):
        super().__init__()
        self.k = k
        self.selector = nn.Sequential(
            nn.Conv2d(1536, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512,  128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128,  1,   kernel_size=3, padding=1),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, local_feats, cls_token):
        """
        local_feats : (N, 768, 7, 7)
        cls_token   : (768,)
        """
        N = local_feats.shape[0]

        # Expand CLS token spatially: (768,) → (N, 768, 7, 7)
        global_expanded = cls_token.view(1, VIT_DIM, 1, 1).expand(N, -1, 7, 7)

        # Concatenate → (N, 1536, 7, 7)
        obj_scene = torch.cat([local_feats, global_expanded], dim=1)

        scores = self.selector(obj_scene)       # (N, 1, 7, 7)
        scores = self.gap(scores).view(N)       # (N,)
        scores = torch.softmax(scores, dim=0)   # (N,)

        actual_k     = min(self.k, N)
        topk_indices = torch.topk(scores, actual_k).indices
        topk_tensors = obj_scene[topk_indices]  # (actual_k, 1536, 7, 7)

        if actual_k < self.k:
            pad = torch.zeros(
                self.k - actual_k, topk_tensors.shape[1],
                topk_tensors.shape[2], topk_tensors.shape[3],
                device=topk_tensors.device
            )
            topk_tensors = torch.cat([topk_tensors, pad], dim=0)

        return topk_tensors  # (k, 1536, 7, 7)


class PredictionHead(nn.Module):
    def __init__(self, k=10, in_channels=1536, num_actions=4, num_reasons=21):
        super().__init__()
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        feat_dim     = k * in_channels  # 10 * 1536 = 15360

        self.shared_fc = nn.Sequential(
            nn.Linear(feat_dim, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.action_head = nn.Linear(1024, num_actions)
        self.reason_head = nn.Linear(1024, num_reasons)

    def forward(self, topk_tensors):
        pooled = self.gap(topk_tensors)    # (k, 1536, 1, 1)
        flat   = pooled.view(-1)           # (15360,)
        feat   = self.shared_fc(flat)      # (1024,)
        return self.action_head(feat), self.reason_head(feat)


class OIANetViT(nn.Module):
    def __init__(self, k=TOP_K):
        super().__init__()
        self.aism            = AISM(k=k)
        self.prediction_head = PredictionHead(k=k)

    def forward(self, cls_tokens, patch_tokens_batch, boxes_list):
        """
        cls_tokens         : (B, 768)
        patch_tokens_batch : (B, 256, 768)
        boxes_list         : list of B tensors, each (N_i, 4)
        """
        action_logits_list, reason_logits_list = [], []

        for i in range(len(boxes_list)):
            cls_i          = cls_tokens[i]           # (768,)
            patch_tokens_i = patch_tokens_batch[i]   # (256, 768)
            boxes_i        = boxes_list[i].to(device) # (N, 4)

            local_i = extract_roi_features(patch_tokens_i, boxes_i)  # (N, 768, 7, 7)

            topk = self.aism(local_i, cls_i)
            a, r = self.prediction_head(topk)
            action_logits_list.append(a)
            reason_logits_list.append(r)

        return torch.stack(action_logits_list), torch.stack(reason_logits_list)

# ── Training loop ─────────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train() if train else model.eval()
    total_loss = action_loss_total = reason_loss_total = 0.0
    n_batches  = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for cls_tokens, patch_tokens, action_labels, reason_labels, boxes_list in loader:
            cls_tokens    = cls_tokens.to(device)
            patch_tokens  = patch_tokens.to(device)
            action_labels = action_labels.to(device)
            reason_labels = reason_labels.to(device)

            action_logits, reason_logits = model(cls_tokens, patch_tokens, boxes_list)

            l_action = criterion(action_logits, action_labels)
            l_reason = criterion(reason_logits, reason_labels)
            loss     = l_action + LAMBDA * l_reason

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss        += loss.item()
            action_loss_total += l_action.item()
            reason_loss_total += l_reason.item()
            n_batches         += 1

    return (total_loss / n_batches,
            action_loss_total / n_batches,
            reason_loss_total / n_batches)

# ── Main ──────────────────────────────────────────────────────────────────────
print("Loading datasets...")
train_dataset = BDDOIADataset("train")
val_dataset   = BDDOIADataset("val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)

model     = OIANetViT(k=TOP_K).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=LR_STEP, gamma=0.1)

print(f"\nDevice       : {device}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Starting training...\n")

log               = []
best_val_loss     = float("inf")
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]  # LR used this epoch

    train_loss, train_a, train_r = run_epoch(model, train_loader, criterion, optimizer, train=True)
    val_loss,   val_a,   val_r   = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train: {train_loss:.4f} (A:{train_a:.4f} R:{train_r:.4f}) | "
          f"Val: {val_loss:.4f} (A:{val_a:.4f} R:{val_r:.4f}) | "
          f"LR: {current_lr:.6f}")

    log.append({
        "epoch"       : epoch,
        "train_loss"  : train_loss,
        "train_action": train_a,
        "train_reason": train_r,
        "val_loss"    : val_loss,
        "val_action"  : val_a,
        "val_reason"  : val_r,
        "lr"          : current_lr,
    })
    with open(LOG_PATH, "w") as f:
        json.dump(log, f, indent=2)

    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        epochs_no_improve = 0
        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss"       : val_loss,
        }, SAVE_PATH)
        print(f"  ✓ Best model saved (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement {epochs_no_improve}/{PATIENCE}")
        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

print(f"\nDone. Best val loss: {best_val_loss:.4f}")
print(f"Model : {SAVE_PATH}")
print(f"Log   : {LOG_PATH}")

Loading datasets...
[train] 16028 samples loaded
[val] 2259 samples loaded

Device       : cuda
Train batches: 1002
Val batches  : 142
Early stopping patience: 7
Starting training...

Epoch 01/50 | Train: 0.9984 (A:0.6686 R:0.3298) | Val: 0.9105 (A:0.5985 R:0.3119) | LR: 0.001000
  ✓ Best model saved (val_loss=0.9105)
Epoch 02/50 | Train: 0.9372 (A:0.6164 R:0.3208) | Val: 0.8947 (A:0.5822 R:0.3126) | LR: 0.001000
  ✓ Best model saved (val_loss=0.8947)
Epoch 03/50 | Train: 0.9278 (A:0.6088 R:0.3190) | Val: 0.9081 (A:0.5871 R:0.3209) | LR: 0.001000
  No improvement 1/7
Epoch 04/50 | Train: 0.9204 (A:0.6041 R:0.3162) | Val: 0.8846 (A:0.5840 R:0.3005) | LR: 0.001000
  ✓ Best model saved (val_loss=0.8846)
Epoch 05/50 | Train: 0.9178 (A:0.6031 R:0.3147) | Val: 0.8819 (A:0.5824 R:0.2995) | LR: 0.001000
  ✓ Best model saved (val_loss=0.8819)
Epoch 06/50 | Train: 0.9181 (A:0.6035 R:0.3146) | Val: 0.8822 (A:0.5809 R:0.3013) | LR: 0.001000
  No improvement 1/7
Epoch 07/50 | Train: 0.9193 (A:0.604